# Keypoint-модель для бедра (большой вертел, шейка бедра, седалищная кость)

Ноутбук для Kaggle/Colab. Обучает heatmap-regression модель (Шаг 2 плана,
docs/plan.md) находить 3 именованные точки на снимках бедра. Backbone —
тот же подход, что в Step 1 (ResNet18, ImageNet pretrained, частичная
заморозка), но вместо classification-головы — decoder, восстанавливающий
heatmap того же размера, что вход (224x224), по одному каналу на точку.

## Перед запуском

Загрузите `training_package` (тот же, что для Step 1, там уже есть все
нужные PNG) + файл `hip_keypoints_manifest.csv` (координаты точек и
полигона малого вертела, полученные из экспорта CVAT). Пропишите пути
в ячейке конфигурации.

## Важно про координаты

В `hip_keypoints_manifest.csv` координаты — в пиксельах ИСХОДНОГО снимка
(width/height у каждой строки свои, снимки разного размера). При ресайзе
до 224x224 координаты нужно масштабировать пропорционально — это делает
albumentations автоматически через `keypoint_params`, вручную ничего
пересчитывать не нужно.

## Воспроизводимость

Тот же набор мер, что в Step 1: seed для Python/NumPy/PyTorch/CUDA/
albumentations, детерминированные алгоритмы cuDNN, group k-fold по
study_uid (те же fold, что уже посчитаны для Step 1 — переиспользуем
колонку `fold` из hip_keypoints_manifest.csv, не пересчитываем заново).


In [ ]:
!pip install -q albumentations timm scikit-learn


In [ ]:
import os
import random
import math
import json

import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import accuracy_score


## Воспроизводимость: seed + детерминизм

In [ ]:
SEED = 42


def set_full_determinism(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_full_determinism(SEED)


def worker_init_fn(worker_id: int) -> None:
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_generator(seed: int = SEED) -> torch.Generator:
    g = torch.Generator()
    g.manual_seed(seed)
    return g


## Конфигурация

In [ ]:
# --- ПРАВЬТЕ ПОД СВОЮ СРЕДУ ---
IMAGES_DIR = "/kaggle/input/datasets/cgvhbjmk/dxa-data/images"
KEYPOINTS_MANIFEST_PATH = "/kaggle/input/datasets/cgvhbjmk/dxa-data/hip_keypoints_manifest.csv"
OUTPUT_DIR = "./outputs_keypoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 10
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
N_FOLDS = 5
TEST_FOLD = N_FOLDS - 1  # тот же принцип, что в Step 1 — честный отложенный тест
NUM_WORKERS = 2

KEYPOINT_NAMES = ["большой_вертел", "шейка_бедра", "седалищная_кость"]
HEATMAP_SIGMA = 6.0  # пикселей, в масштабе 224x224
VISIBILITY_THRESHOLD = 0.3  # порог высоты пика heatmap для решения "точка видна"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Аугментация

Геометрические трансформации (поворот/масштаб/сдвиг/флип) применяются
через `A.Compose(..., keypoint_params=...)` — albumentations сама
пересчитывает координаты точек согласованно с преобразованием картинки,
вручную ничего не трансформируем.

`CoarseDropout` НЕ используется (в отличие от Step 1) — риск случайно
закрасить именно ту точку, которую модель должна найти, контрпродуктивен
для keypoint-задачи.


In [ ]:
def clahe(image: np.ndarray) -> np.ndarray:
    return cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(image)


def build_train_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
        A.RandomGamma(gamma_limit=(70, 130), p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.08), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
        ], p=0.4),
        A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), rotate=(-15, 15), fit_output=False, p=0.7),
        A.HorizontalFlip(p=0.5),
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ], keypoint_params=A.KeypointParams(format="xy", remove_invisible=False), seed=SEED)


def build_eval_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ], keypoint_params=A.KeypointParams(format="xy", remove_invisible=False))


## Dataset: генерация heatmap на лету

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

COORD_COLS = [
    ("bolshoy_vertel_x", "bolshoy_vertel_y"),
    ("sheyka_bedra_x", "sheyka_bedra_y"),
    ("sedalishnaya_kost_x", "sedalishnaya_kost_y"),
]


def make_gaussian_heatmap(size: int, x: float, y: float, sigma: float) -> np.ndarray:
    yy, xx = np.mgrid[0:size, 0:size]
    heatmap = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    return heatmap.astype(np.float32)


def get_coords_and_visibility(row):
    # Отсутствующая точка (NaN) — это НЕ грязные данные, а содержательный
    # сигнал: разметчик не увидел структуру на кадре, а критерий "видна
    # ли анатомия" (ТЗ, положение бедра) — это буквально то, чему должна
    # научиться модель. Поэтому такие точки НЕ выбрасываются из выборки,
    # а учат модель предсказывать "структуры здесь нет" (нулевая heatmap).
    coords, visible = [], []
    for xcol, ycol in COORD_COLS:
        x, y = row[xcol], row[ycol]
        if pd.isna(x) or pd.isna(y):
            coords.append(None)
            visible.append(0)
        else:
            coords.append((float(x), float(y)))
            visible.append(1)
    return coords, visible


CROP_AUG_MARGIN_MIN_PX = 5
CROP_AUG_MARGIN_MAX_PX = 25


CROP_DUPLICATE_FRACTION = 0.3  # доля строк, для которых добавляем ещё и обрезанную копию


def expand_with_crop_duplicates(df: pd.DataFrame, fraction: float = CROP_DUPLICATE_FRACTION, seed: int = SEED) -> pd.DataFrame:
    """Явное дополнение обучающей выборки: ВСЕ строки остаются как есть
    (force_crop=0), и ДОПОЛНИТЕЛЬНО для случайной доли строк (fraction)
    добавляется ещё одна, обрезанная копия (force_crop=1) — не замена,
    а именно добавление, в духе обычной аугментации.

    Доля обрезанных копий сознательно МЕНЬШЕ половины (не 1:1 clean:crop):
    в реальных данных на инференсе подавляющее большинство снимков —
    корректные, без дефекта обрезки. Если давать модели поровну чистых и
    обрезанных примеров, она станет непропорционально "пугливой" и будет
    ошибочно занижать уверенность даже на нормальных снимках (это и
    произошло в прошлой версии с вероятностной аугментацией p=0.3 —
    результат был плохим именно на полностью видимых тестовых снимках).
    Здесь то же соотношение (30%), но детерминированно фиксированное на
    весь датасет, а не вероятностно на каждую эпоху.
    """
    clean = df.copy()
    clean["force_crop"] = 0

    subset = df.sample(frac=fraction, random_state=seed)
    cropped = subset.copy()
    cropped["force_crop"] = 1

    return pd.concat([clean, cropped], ignore_index=True)


def force_crop_all(df: pd.DataFrame) -> pd.DataFrame:
    """Копия df, где ВСЕ строки помечены force_crop=1 — не для обучения,
    а для синтетической ПРОВЕРКИ детекции невидимости на val/test.

    Естественных невидимых точек в разметке единицы (5 на всю CV-
    валидацию, 0 на отложенном тесте) — ни калибровать порог, ни
    проверять его на честном тесте по ним статистически нельзя. Эта
    функция даёт полноразмерный, гарантированно содержащий обрезки
    набор, оцениваемый ОТДЕЛЬНО от натуральных (неаугментированных)
    метрик — не подменяет их, а проверяет то, что натуральные данные
    проверить не могут."""
    out = df.copy()
    out["force_crop"] = 1
    return out


def apply_crop_augmentation(image: np.ndarray, coords_224: list, visible: list):
    """Имитирует реальный дефект укладки — часть анатомии обрезана краем
    поля скана.

    ВАЖНО (проверено на реальных координатах датасета): три наши точки
    лежат в разных зонах кадра — большой_вертел/шейка_бедра почти всегда
    в верхней половине (y_frac до ~0.56), седалищная_кость — в основном
    ниже середины (y_frac медиана 0.58). Если брать случайный край +
    случайную глубину (как раньше), значительная доля аугментированных
    примеров НИЧЕГО не закрывает — полоса рисуется, но ни одна точка в
    неё не попадает, и обучающего сигнала не возникает.

    Поэтому глубина обрезки не случайна, а считается от РЕАЛЬНОЙ видимой
    точки этого же снимка (случайно выбранной из видимых) — так, чтобы
    полоса её гарантированно накрыла (+ запас CROP_AUG_MARGIN_*_PX).
    Край выбирается из двух ближайших к этой точке — так глубина
    остаётся реалистичной (не приходится закрашивать половину снимка,
    чтобы дотянуться до точки у дальнего края).
    """
    size = image.shape[0]

    visible_idx = [i for i, v in enumerate(visible) if v == 1 and coords_224[i] is not None]
    if not visible_idx:
        return image, coords_224, visible  # нечего гарантированно закрывать — все точки и так невидимы

    target_i = random.choice(visible_idx)
    tx, ty = coords_224[target_i]
    margin = random.uniform(CROP_AUG_MARGIN_MIN_PX, CROP_AUG_MARGIN_MAX_PX)

    dist_to_edge = {"top": ty, "bottom": size - ty, "left": tx, "right": size - tx}
    two_nearest = sorted(dist_to_edge, key=dist_to_edge.get)[:2]
    edge = random.choice(two_nearest)
    depth = int(min(size, dist_to_edge[edge] + margin))

    image = image.copy()
    if edge == "top":
        image[:depth, :] = 0
        in_strip = lambda x, y: y < depth
    elif edge == "bottom":
        image[size - depth:, :] = 0
        in_strip = lambda x, y: y > size - depth
    elif edge == "left":
        image[:, :depth] = 0
        in_strip = lambda x, y: x < depth
    else:
        image[:, size - depth:] = 0
        in_strip = lambda x, y: x > size - depth

    new_visible = list(visible)
    new_coords = list(coords_224)
    for i, c in enumerate(coords_224):
        if c is not None and in_strip(c[0], c[1]):
            new_visible[i] = 0
            new_coords[i] = None  # heatmap для неё будет нулевой, как для обычной "невидимой" точки

    return image, new_coords, new_visible


class HipKeypointDataset(Dataset):
    def __init__(self, df: pd.DataFrame, images_dir: str, train: bool, force_crop_eval: bool = False):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.train = train
        # force_crop_eval — синтетическая проверка на val/test (force_crop_all),
        # не аугментация обучения: применяет обрезку и в режиме train=False
        self.force_crop_eval = force_crop_eval

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.images_dir, row["image_filename"])
        image_u8 = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if image_u8 is None:
            raise FileNotFoundError(f"Не удалось прочитать {path}")
        image_u8 = clahe(image_u8)

        coords, visible = get_coords_and_visibility(row)
        present_idx = [i for i, c in enumerate(coords) if c is not None]
        present_pts = [coords[i] for i in present_idx]

        transform = build_train_transform() if self.train else build_eval_transform()
        transformed = transform(image=image_u8, keypoints=present_pts)
        image_t = transformed["image"]
        pts_t = transformed["keypoints"]

        # раскладываем трансформированные точки обратно по их исходным
        # индексам-каналам; для отсутствующих оставляем None -> нулевая heatmap
        coords_224 = [None, None, None]
        for local_i, orig_i in enumerate(present_idx):
            coords_224[orig_i] = pts_t[local_i]

        if (self.train or self.force_crop_eval) and int(row.get("force_crop", 0)) == 1:
            image_t, coords_224, visible = apply_crop_augmentation(image_t, coords_224, visible)

        heatmaps = np.stack([
            make_gaussian_heatmap(IMAGE_SIZE, c[0], c[1], HEATMAP_SIGMA) if c is not None
            else np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
            for c in coords_224
        ], axis=0)

        rgb = np.stack([image_t] * 3, axis=-1).astype(np.float32) / 255.0
        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        image_tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).float()
        heatmap_tensor = torch.from_numpy(heatmaps).float()

        # -1,-1 плейсхолдер для отсутствующих точек — не участвует в
        # метрике пиксельной ошибки (маскируется через visible_tensor)
        coords_tensor = torch.tensor(
            [c if c is not None else (-1.0, -1.0) for c in coords_224], dtype=torch.float32
        )
        visible_tensor = torch.tensor(visible, dtype=torch.float32)

        return image_tensor, heatmap_tensor, coords_tensor, visible_tensor


manifest = pd.read_csv(KEYPOINTS_MANIFEST_PATH)
coord_cols_flat = [c for pair in COORD_COLS for c in pair]
n_fully_visible = manifest[coord_cols_flat].notna().all(axis=1).sum()
print(f"Строк всего: {len(manifest)}, со всеми 3 точками видимыми: {n_fully_visible}")
print("Строки с хотя бы одной невидимой точкой НЕ выбрасываются — это тоже обучающий сигнал.")
for xcol, ycol in COORD_COLS:
    n_missing = manifest[xcol].isna().sum()
    print(f"  {xcol.rsplit(chr(95),1)[0]}: невидима на {n_missing} снимках")

cv_manifest = manifest[manifest["fold"] != TEST_FOLD].reset_index(drop=True)
test_manifest = manifest[manifest["fold"] == TEST_FOLD].reset_index(drop=True)
print()
print(f"CV: {len(cv_manifest)}, отложенный тест: {len(test_manifest)}")


## Модель: ResNet18 encoder + decoder до heatmap

In [ ]:
class KeypointHeatmapModel(nn.Module):
    def __init__(self, num_keypoints: int = 3):
        super().__init__()
        backbone = timm.create_model("resnet18", pretrained=True, features_only=True, out_indices=(4,))
        self.encoder = backbone  # выход: (B, 512, 7, 7) для входа 224x224

        for name, param in self.encoder.named_parameters():
            if name.startswith(("conv1", "bn1", "layer1", "layer2")):
                param.requires_grad = False

        # 5 блоков апсемпла x2: 7 -> 14 -> 28 -> 56 -> 112 -> 224
        channels = [512, 256, 128, 64, 32, 16]
        blocks = []
        for i in range(5):
            blocks.append(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False))
            blocks.append(nn.Conv2d(channels[i], channels[i + 1], kernel_size=3, padding=1))
            blocks.append(nn.BatchNorm2d(channels[i + 1]))
            blocks.append(nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(*blocks)
        self.head = nn.Conv2d(channels[-1], num_keypoints, kernel_size=1)

    def forward(self, x):
        feat = self.encoder(x)[0]
        decoded = self.decoder(feat)
        return self.head(decoded)


def build_model(num_keypoints: int = 3) -> nn.Module:
    return KeypointHeatmapModel(num_keypoints)


def build_optimizer(model: nn.Module):
    head_params = list(model.decoder.parameters()) + list(model.head.parameters())
    backbone_params = [p for p in model.encoder.parameters() if p.requires_grad]
    return torch.optim.AdamW([
        {"params": head_params, "lr": HEAD_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ])


def heatmap_to_coords(heatmap: torch.Tensor) -> torch.Tensor:
    # heatmap: (B, K, H, W) -> (B, K, 2) координаты argmax
    b, k, h, w = heatmap.shape
    flat = heatmap.view(b, k, -1)
    idx = flat.argmax(dim=-1)
    ys = (idx // w).float()
    xs = (idx % w).float()
    return torch.stack([xs, ys], dim=-1)


# Обычный MSE на heatmap с крошечным гауссовым пятном на фоне нулей даёт
# сети тривиальное "хорошее" решение — предсказывать почти всюду ноль
# (98% таргета и так нули). Координата (argmax) при этом может остаться
# верной, а вот абсолютная высота пика уходит к нулю — ломает проверку
# видимости по порогу. Взвешиваем ошибку сильнее в районе истинной точки,
# чтобы сеть не могла "спрятаться" в тривиальном решении.
POS_WEIGHT = 25.0


def weighted_heatmap_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    weights = 1.0 + POS_WEIGHT * target
    return (weights * (pred - target) ** 2).mean()


## Цикл обучения

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_pixel_error = 0.0
    n_visible_points = 0
    visibility_correct = 0
    n_points_total = 0

    with torch.set_grad_enabled(is_train):
        for images, heatmaps, kp_true, visible in loader:
            images = images.to(DEVICE)
            heatmaps = heatmaps.to(DEVICE)

            pred_heatmaps = model(images)
            loss = criterion(pred_heatmaps, heatmaps)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            pred_heatmaps_cpu = pred_heatmaps.detach().cpu()
            pred_coords = heatmap_to_coords(pred_heatmaps_cpu)

            # пиксельная ошибка — только по точкам, которые реально видны
            # (для невидимых кет истинных координат нет, сравнивать не с чем)
            mask = visible.bool()
            if mask.any():
                err = (pred_coords - kp_true).pow(2).sum(-1).sqrt()
                total_pixel_error += err[mask].sum().item()
                n_visible_points += mask.sum().item()

            # точность предсказания "видна ли точка вообще" — по высоте пика heatmap
            peak_values = pred_heatmaps_cpu.flatten(2).max(dim=-1).values  # (B, K)
            pred_visible = (peak_values > VISIBILITY_THRESHOLD).float()
            visibility_correct += (pred_visible == visible).sum().item()
            n_points_total += visible.numel()

            bs = images.size(0)
            total_loss += loss.item() * bs

    avg_loss = total_loss / len(loader.dataset)
    avg_pixel_error = total_pixel_error / n_visible_points if n_visible_points > 0 else float("nan")
    visibility_acc = visibility_correct / n_points_total
    return avg_loss, avg_pixel_error, visibility_acc


def train_one_fold(fold: int, cv_manifest: pd.DataFrame):
    set_full_determinism(int(SEED + fold))

    train_df = cv_manifest[cv_manifest["fold"] != fold]
    val_df = cv_manifest[cv_manifest["fold"] == fold]

    # val НЕ дополняем обрезанными копиями — оценка должна остаться честной,
    # на реальных (не синтетически испорченных) данных
    train_df = expand_with_crop_duplicates(train_df)

    train_ds = HipKeypointDataset(train_df, IMAGES_DIR, train=True)
    val_ds = HipKeypointDataset(val_df, IMAGES_DIR, train=False)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        drop_last=True, worker_init_fn=worker_init_fn, generator=make_generator(int(SEED + fold)),
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = build_model().to(DEVICE)
    optimizer = build_optimizer(model)
    criterion = weighted_heatmap_loss

    best_val_error = float("inf")
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(NUM_EPOCHS):
        train_loss, train_err, train_vis_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_err, val_vis_acc = run_epoch(model, val_loader, criterion)

        history.append(dict(epoch=epoch + 1, train_loss=train_loss, train_pixel_error=train_err,
                             train_visibility_acc=train_vis_acc, val_loss=val_loss,
                             val_pixel_error=val_err, val_visibility_acc=val_vis_acc))
        print(f"fold {fold} | epoch {epoch+1}/{NUM_EPOCHS} | "
              f"train_loss={train_loss:.5f} train_err={train_err:.2f}px train_vis_acc={train_vis_acc:.3f} | "
              f"val_loss={val_loss:.5f} val_err={val_err:.2f}px val_vis_acc={val_vis_acc:.3f}")

        # модель выбирается по loss (учитывает и heatmap-качество, и
        # неявно visibility через нулевые таргеты), а не только по pixel_error,
        # который для фолдов с редкими "невидимыми" случаями может быть NaN/шумным
        if val_loss < best_val_error:
            best_val_error = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"fold {fold}: early stopping на эпохе {epoch+1}")
                break

    checkpoint = {
        "model_state_dict": best_state,
        "keypoint_names": KEYPOINT_NAMES,
        "image_size": IMAGE_SIZE,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_error,
        "fold": fold,
        "seed": SEED,
    }
    torch.save(checkpoint, os.path.join(OUTPUT_DIR, f"hip_keypoints_fold{fold}.pt"))
    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, f"kp_history_fold{fold}.csv"), index=False)

    return best_val_error


## Запуск CV (folds 0..TEST_FOLD-1)

In [ ]:
fold_losses = []
for fold in sorted(int(f) for f in cv_manifest["fold"].unique()):
    loss = train_one_fold(fold, cv_manifest)
    fold_losses.append(loss)
    print()
    print(f"=== fold {fold}: best val loss = {loss:.5f} ===")
    print()

# ВАЖНО: это val_loss (критерий отбора чекпоинта), НЕ ошибка в пикселях —
# реальную пиксельную ошибку и точность видимости смотрите в логах по
# эпохам выше (val_err/val_vis_acc) или в kp_history_fold{N}.csv
print("val_loss по фолдам:", [round(l, 5) for l in fold_losses])
print("Среднее:", round(np.mean(fold_losses), 5), "± std:", round(np.std(fold_losses), 5))


## Калибровка порога видимости

`VISIBILITY_THRESHOLD=0.3` в конфигурации был угадан, не измерен. Подбираем порог честно — на CV-валидации, где каждый фолд оценивается СВОЕЙ собственной моделью (той, что не видела эти данные при обучении) — без утечки, никакого доступа к отложенному тесту тут нет.

**Важно**: естественных невидимых точек в CV-валидации всего ~5 (из ~369
возможных) — калибровка ТОЛЬКО по ним статистически ненадёжна (порог,
подобранный по 5 примерам, легко переобучается на шум). Поэтому ниже
порог подбирается на ОБЪЕДИНЕНИИ натуральной валидации + синтетической
проверки (`force_crop_all` — та же обрезка краёв, что и в аугментации
обучения, применённая ко ВСЕЙ копии val, а не к случайной доле) — это
даёт статистически значимое число настоящих отрицательных примеров, не
теряя при этом естественное распределение (обе части взвешены равно по
числу точек, не искусственно смещены).

In [ ]:
def collect_cv_val_predictions(cv_manifest: pd.DataFrame, force_crop_eval: bool = False):
    all_true, all_peak = [], []
    for fold in sorted(int(f) for f in cv_manifest["fold"].unique()):
        val_df = cv_manifest[cv_manifest["fold"] == fold]
        if force_crop_eval:
            val_df = force_crop_all(val_df)
        val_ds = HipKeypointDataset(val_df, IMAGES_DIR, train=False, force_crop_eval=force_crop_eval)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        ckpt = torch.load(os.path.join(OUTPUT_DIR, f"hip_keypoints_fold{fold}.pt"), map_location=DEVICE, weights_only=False)
        m = build_model().to(DEVICE)
        m.load_state_dict(ckpt["model_state_dict"])
        m.eval()

        with torch.no_grad():
            for images, _heatmaps, _kp_true, visible in val_loader:
                images = images.to(DEVICE)
                preds = m(images).cpu()
                peak_values = preds.flatten(2).max(dim=-1).values
                all_true.append(visible.numpy())
                all_peak.append(peak_values.numpy())

    return np.concatenate(all_true), np.concatenate(all_peak)


set_full_determinism(SEED)  # фиксируем, какой край/глубину обрежет force_crop_all — воспроизводимая калибровка

true_vis_natural, peak_natural = collect_cv_val_predictions(cv_manifest, force_crop_eval=False)
true_vis_synth, peak_synth = collect_cv_val_predictions(cv_manifest, force_crop_eval=True)

true_vis = np.concatenate([true_vis_natural, true_vis_synth])
peak_vals = np.concatenate([peak_natural, peak_synth])

print(f"Натуральная валидация: {len(true_vis_natural)} точек, видимых {int(true_vis_natural.sum())} "
      f"(невидимых всего {int((1 - true_vis_natural).sum())} — статистически мало для калибровки в одиночку)")
print(f"Синтетическая проверка (force_crop_all): {len(true_vis_synth)} точек, видимых {int(true_vis_synth.sum())}, "
      f"невидимых {int((1 - true_vis_synth).sum())}")

candidate_thresholds = np.arange(0.02, 0.95, 0.02)
best_threshold, best_acc = None, -1.0
for thr in candidate_thresholds:
    pred = (peak_vals > thr).astype(np.float32)
    acc = (pred == true_vis).mean()
    if acc > best_acc:
        best_acc = float(acc)
        best_threshold = float(thr)

acc_at_default = ((peak_vals > VISIBILITY_THRESHOLD).astype(np.float32) == true_vis).mean()
print(f"\nТочность при старом пороге {VISIBILITY_THRESHOLD} (натуральная+синтетика): {acc_at_default:.3f}")
print(f"Откалиброванный порог: {best_threshold:.2f}, точность (натуральная+синтетика): {best_acc:.3f}")

# отдельно смотрим, как откалиброванный порог ведёт себя на КАЖДОЙ части
# по отдельности — чтобы не пропустить перекос в одну сторону
acc_natural_only = ((peak_natural > best_threshold).astype(np.float32) == true_vis_natural).mean()
acc_synth_only = ((peak_synth > best_threshold).astype(np.float32) == true_vis_synth).mean()
print(f"  из них: точность на натуральной валидации = {acc_natural_only:.3f}, "
      f"на синтетической обрезке = {acc_synth_only:.3f}")

VISIBILITY_THRESHOLD = best_threshold
print()
print(f"VISIBILITY_THRESHOLD обновлён на {VISIBILITY_THRESHOLD:.2f} — используется в ячейке теста ниже")


## Финальная оценка на отложенном тесте

Средняя ошибка в пикселях (масштаб 224x224). Для примерного перевода
в миллиметры: снимки исходно ~280x300px при PixelSpacing 1.05x0.6мм
(ответ организаторов) — но так как мы ресайзим к 224, масштаб на снимок
свой, поэтому здесь только пиксельная ошибка на входе сети, без точного
перевода в мм (это отдельная задача при интеграции в rule engine, где
конвертация будет делаться для каждого снимка индивидуально по его
исходному размеру).


In [ ]:
ensemble_models = []
for fold in sorted(int(f) for f in cv_manifest["fold"].unique()):
    ckpt = torch.load(os.path.join(OUTPUT_DIR, f"hip_keypoints_fold{fold}.pt"), map_location=DEVICE, weights_only=False)
    m = build_model().to(DEVICE)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    ensemble_models.append(m)


def run_ensemble_test(df: pd.DataFrame, force_crop_eval: bool = False):
    ds = HipKeypointDataset(df, IMAGES_DIR, train=False, force_crop_eval=force_crop_eval)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    per_kp_errors = {name: [] for name in KEYPOINT_NAMES}
    visibility_true = {name: [] for name in KEYPOINT_NAMES}
    visibility_pred = {name: [] for name in KEYPOINT_NAMES}

    with torch.no_grad():
        for images, heatmaps, kp_true, visible in loader:
            images = images.to(DEVICE)
            preds = torch.stack([m(images).cpu() for m in ensemble_models], dim=0).mean(dim=0)
            pred_coords = heatmap_to_coords(preds)
            errors = (pred_coords - kp_true).pow(2).sum(-1).sqrt()
            peak_values = preds.flatten(2).max(dim=-1).values
            pred_vis = (peak_values > VISIBILITY_THRESHOLD).float()

            for k, name in enumerate(KEYPOINT_NAMES):
                vis_mask = visible[:, k].bool()
                if vis_mask.any():
                    per_kp_errors[name].extend(errors[vis_mask, k].tolist())
                visibility_true[name].extend(visible[:, k].tolist())
                visibility_pred[name].extend(pred_vis[:, k].tolist())

    return per_kp_errors, visibility_true, visibility_pred


print(f"Отложенный тест (натуральный, без изменений): {len(test_manifest)} изображений")
per_kp_errors, visibility_true, visibility_pred = run_ensemble_test(test_manifest, force_crop_eval=False)
for name in KEYPOINT_NAMES:
    errs = np.array(per_kp_errors[name])
    vis_acc = accuracy_score(visibility_true[name], visibility_pred[name])
    n_visible = int(sum(visibility_true[name]))
    n_total = len(visibility_true[name])
    if len(errs) > 0:
        print(f"{name}: MAE={errs.mean():.2f}px (n={len(errs)} видимых), "
              f"median={np.median(errs):.2f}px, max={errs.max():.2f}px | "
              f"точность определения видимости={vis_acc:.3f} (видимых в тесте: {n_visible}/{n_total})")
    else:
        print(f"{name}: нет видимых точек в отложенном тесте для расчёта MAE | "
              f"точность определения видимости={vis_acc:.3f}")
    if n_visible == n_total:
        print(f"  (все {n_total} точек видимы — эта точность НЕ проверяет детекцию невидимости, "
              f"см. синтетическую проверку ниже)")

print()
print("Синтетическая проверка детекции невидимости (force_crop_all на копии отложенного теста, "
      "НИКОГДА не участвовавшей ни в обучении, ни в подборе порога):")
set_full_determinism(SEED)  # воспроизводимый набор обрезок для отчёта
per_kp_errors_s, visibility_true_s, visibility_pred_s = run_ensemble_test(force_crop_all(test_manifest), force_crop_eval=True)
for name in KEYPOINT_NAMES:
    vis_acc_s = accuracy_score(visibility_true_s[name], visibility_pred_s[name])
    n_visible_s = int(sum(visibility_true_s[name]))
    n_total_s = len(visibility_true_s[name])
    print(f"{name}: точность определения видимости на синтетической обрезке={vis_acc_s:.3f} "
          f"(видимых {n_visible_s}/{n_total_s}, т.е. невидимых {n_total_s - n_visible_s} — "
          f"именно эта часть и проверяет детекцию обрезки)")


## Что дальше

- Веса каждого фолда: `hip_keypoints_fold{N}.pt` в `./outputs_keypoints/`
- Ансамбль (усреднение heatmap-предсказаний всех фолдов) уже используется
  в финальной оценке выше
- Следующий шаг — интеграция в rule engine (docs/plan.md, Шаг 3):
  перевод координат точек в геометрические критерии (видимость структур,
  вход в расчёт ROI-отступов вместе с классической математикой по краю
  поля скана)
